Taking the Features spreadsheet and plotting sampled imagary and cross sections for each entry

In [ ]:
#Import packages
import os
os.chdir("/mnt/export/lee/1-Projects/deepcaldera") #the directory that holds the scripts, output_files, and source_data directories.
import numpy as np
import pandas as pd
import openpyxl
from pathlib import Path
import rasterio

In [ ]:
#Read the dataframe
df = pd.read_excel("output_files/SF3 - Features_dataset.xlsx",index_col=0)
df.head()
df = df.rename(columns={"Feature_Diameter_km":"Diameter (km)"})
df2 = df.copy() # a backup

In [ ]:
#read the gebco map. this is slow as the file is big.
filename = Path("source_data/gebco/gebco_2023_clipped_to_seamounts_proj.tif")
src = rasterio.open(filename)
src_data= src.read(1)

In [ ]:
#import the file with the sampling/plotting functions
%run scripts/caldera.py

#setup a test for the plot_features function.
n="E-3"
row = df.loc[n]
row2 = pd.Series({"Long":row["Long"], "Lat":row["Lat"],"Diameter (km)":row["Caldera_Diameter_km"]})
print(row, row2)
feat = plot_features(row, src_data,src, dim=256,row2=None, second=False)
plt.figtext(-0.25,1.0,n, weight='bold', transform=plt.gca().transAxes)
plt.legend(["cross section S-N","cross section SW-NE","cross section W-E","cross section NW-SE","mean depth at feature edge", "depth range at feature edge"])


In [ ]:
#second test
_ = plot_features(pos.iloc[100], src_data, src, dim=256)

In [ ]:
# A truncated sample, printed to the screen.
from matplotlib.backends.backend_pdf import PdfPages
%run scripts/caldera.py
from matplotlib.lines import Line2D

# Create the PdfPages object to which we will save the pages:
# The with statement makes sure that the PdfPages object is closed properly at
# the end of the block, even if an Exception occurs.
pcount=0
nrow=3
#print(df.columns
df = df2.copy()[8:11]
df = df.sort_index()

caption=False
while (pcount*nrow)<len(df):
    print(pcount)
    my_nrow = min(len(df)-(pcount*nrow),nrow)
    fig, axs = plt.subplots(my_nrow,2,figsize=(2*4,my_nrow*3.7))
    for i,a in enumerate(axs):
        if pcount*nrow+i >= len(df):
            continue
        row = df.iloc[pcount*nrow+i]
        if (row["Caldera_Diameter_km"]=="<Null>"):
            plot_features(row, src_data, src,axs=a)
        else:
            row2 = pd.Series({"Long":row["Long"], "Lat":row["Lat"],"Diameter (km)":row["Caldera_Diameter_km"]})
            plot_features(row, src_data,src, dim=256,row2=row2,axs=a, second=False)
        a[0].text(1.25,1.0,row.name, weight='bold', transform=a[0].transAxes)
        a[0].set_aspect('equal')
        a[1].set_aspect('auto')
    if not caption:
            caption = True
            # Create custom lines
            prop_cycler = plt.rcParams['axes.prop_cycle']
            # Extract just the colors
            colors = [d['color'] for d in prop_cycler]

            labels = ["cross section S-N",
                             "cross section SW-NE",
                             "cross section W-E",
                             "cross section NW-SE",
                             "mean depth at feature edge",
                             "depth range at feature edge",
                             "mean depth at caldera edge",
                             "depth range at caldera edge"]
            linestyles=["-","-","-","-","-","--","-","--"]
            colors = [colors[0],colors[1],colors[2],colors[3],colors[4],colors[5],colors[5],colors[6],colors[6]]
            lines = [Line2D([0],[0],color=colors[i], linestyle=linestyles[i], label=labels[i]) for i in range(len(labels))]
             
            # Collect legend handles and labels from one subplot
            #handles, labels = axs[0, 0].get_legend_handles_labels()
            
            # Add custom lines to the handles
            handles = lines
            labels = labels
            
            # Add a figure-wide legend at the top
            #print(handles
            fig.legend(handles, labels, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.02))
            
            plt.tight_layout(rect=[0, 0, 1.0, 0.97])  # Leave space for the top legend

            # plt.figlegend(["cross section S-N",
            #                 "cross section SW-NE",
            #                 "cross section W-E",
            #                 "cross section NW-SE",
            #                 "mean depth at feature edge",
            #                 "depth range at feature edge"
            #                 "mean depth at caldera edge",
            #                 "depth range at caldera edge"])
            #pcount=1000
    pcount+=1
#df.to_csv("below_sealevel_overlapping.csv")

In [ ]:
#setting up the supplementary materials plot. We use PdfPages to generate a multipage PDF file,
# then iterate through the excel file and sample/plot each entry
from matplotlib.backends.backend_pdf import PdfPages
%run scripts/caldera.py
#LEGENDS ON ALL PAGES
# Create the PdfPages object to which we will save the pages:
# The with statement makes sure that the PdfPages object is closed properly at
# the end of the block, even if an Exception occurs.
pcount=0
nrow=3
#print(df.columns
df = df2.copy()#
df = df.sort_index()

with PdfPages('S2.pdf') as pdf:
    while (pcount*nrow)<len(df):
        caption=False
        caldera=False
        print(pcount)
        my_nrow = min(len(df)-(pcount*nrow),nrow)
        fig, axs = plt.subplots(nrow,2,figsize=(2*4,nrow*3.7))
        unused = axs.copy()
        for i,a in enumerate(axs):
            if pcount*nrow+i >= len(df):
                a[0].remove()
                a[1].remove()
                continue
            row = df.iloc[pcount*nrow+i]
            if (row["Caldera_Diameter_km"]=="<Null>"):
                plot_features(row, src_data, src,axs=a)
            else:
                row2 = pd.Series({"Long":row["Long"], "Lat":row["Lat"],"Diameter (km)":row["Caldera_Diameter_km"]})
                plot_features(row, src_data,src, dim=256,row2=row2,axs=a, second=False)
                caldera=True
            a[0].text(1.25,1.0,row.name, weight='bold', transform=a[0].transAxes)
            a[0].set_aspect('equal')
            a[1].set_aspect('auto')
        
        if not caption:
            caption = True
            # Create custom lines
            prop_cycler = plt.rcParams['axes.prop_cycle']
            # Extract just the colors
            colors = [d['color'] for d in prop_cycler]

            labels = ["cross section S-N",
                             "cross section SW-NE",
                             "cross section W-E",
                             "cross section NW-SE",
                             "mean depth at feature edge",
                             "depth range at feature edge",
                             "mean depth at caldera edge",
                             "depth range at caldera edge"]
            linestyles=["-","-","-","-","-","--","-","--"]
            colors = [colors[0],colors[1],colors[2],colors[3],colors[4],colors[4],colors[5],colors[5]]
            lines = [Line2D([0],[0],color=colors[i], linestyle=linestyles[i], label=labels[i]) for i in range(len(labels))]
            
            # Collect legend handles and labels from one subplot
            #handles, labels = axs[0, 0].get_legend_handles_labels()
            
            # Add custom lines to the handles
            handles = lines
            labels = labels
            
            # Add a figure-wide legend at the top
            #print(handles
            fig.legend(handles, labels, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 0.98))
            
            plt.tight_layout(rect=[0, 0, 1.0, 0.88])  # Leave space for the top legend
        pdf.savefig()
        plt.close()
        pcount+=1